# Model Cascading

## Learning Objectives
1. Understand cascading as progressive accuracy refinement
2. Implement multi-model cascading with confidence-based routing
3. Analyze accuracy-latency trade-offs in cascade architectures
4. Compare single-model vs cascading strategies

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Level 1: Basic Cascading Decision

In [ ]:
# Level 1: Numpy cascade – confidence-based two-stage routing
# Simulate a fast cheap model and a slow accurate model routing pipeline
np.random.seed(42)

N_SAMPLES = 1000
N_CLASSES = 5

# Synthetic dataset: 1000 requests, each with a true label
true_labels = np.random.randint(0, N_CLASSES, size=N_SAMPLES)

def fast_model_predict(X: np.ndarray) -> tuple:
    """
    Simulated fast model (small 0.5B-style): 30ms latency, 82% accuracy.
    Returns (predicted_label, max_softmax_confidence).
    """
    np.random.seed(99)
    # Generate softmax-like confidences (uniformly noisy, often uncertain)
    raw = np.random.dirichlet(alpha=[0.5] * N_CLASSES, size=len(X))
    preds = raw.argmax(axis=1)
    confs = raw.max(axis=1)
    # Inject 82% accuracy: force correct label for 82% of samples
    correct = np.random.rand(len(X)) < 0.82
    preds[correct] = true_labels[:len(X)][correct]
    return preds, confs

def slow_model_predict(X: np.ndarray) -> tuple:
    """
    Simulated slow model (large 70B-style): 180ms latency, 92% accuracy.
    Returns (predicted_label, max_softmax_confidence).
    """
    np.random.seed(77)
    raw = np.random.dirichlet(alpha=[2.0] * N_CLASSES, size=len(X))
    preds = raw.argmax(axis=1)
    confs = raw.max(axis=1)
    correct = np.random.rand(len(X)) < 0.92
    preds[correct] = true_labels[:len(X)][correct]
    return preds, confs

# Threshold sweep: find optimal cascade threshold
FAST_LATENCY_MS  = 30
SLOW_LATENCY_MS  = 180

thresholds = np.linspace(0.40, 0.95, 20)
results = []

X_dummy = np.zeros((N_SAMPLES, 10))  # placeholder features
fast_preds, fast_confs = fast_model_predict(X_dummy)
slow_preds, _ = slow_model_predict(X_dummy)

for theta in thresholds:
    # Route: high confidence -> fast result, low confidence -> slow result
    escalate_mask = fast_confs < theta
    cascade_preds = fast_preds.copy()
    cascade_preds[escalate_mask] = slow_preds[escalate_mask]

    n_escalated = escalate_mask.sum()
    accuracy = (cascade_preds == true_labels).mean()
    avg_latency = (
        (N_SAMPLES - n_escalated) * FAST_LATENCY_MS +
        n_escalated * (FAST_LATENCY_MS + SLOW_LATENCY_MS)
    ) / N_SAMPLES
    cost = (
        (N_SAMPLES - n_escalated) * 0.001 +
        n_escalated * (0.001 + 0.020)
    ) / N_SAMPLES

    results.append({
        'theta': theta, 'accuracy': accuracy,
        'escalation_rate': n_escalated / N_SAMPLES,
        'avg_latency': avg_latency, 'avg_cost': cost,
    })

# Print table
print('Cascade Threshold Sweep:')
print(f'{"Theta":>7} {"Accuracy":>10} {"Escalation":>12} '
      f'{"Avg Lat(ms)":>13} {"Avg Cost($)":>13}')
print('-' * 60)
for r in results[::3]:
    print(f'{r["theta"]:7.2f} {r["accuracy"]:10.3f} {r["escalation_rate"]:12.1%} '
          f'{r["avg_latency"]:13.1f} {r["avg_cost"]:13.5f}')

# Find Pareto-optimal theta (best accuracy within 60ms average latency budget)
pareto_candidates = [r for r in results if r['avg_latency'] <= 60]
best = max(pareto_candidates, key=lambda r: r['accuracy']) if pareto_candidates else results[0]
print(f'\nBest theta within 60ms budget: {best["theta"]:.2f} '
      f'-> accuracy={best["accuracy"]:.3f}, '
      f'escalation={best["escalation_rate"]:.1%}, '
      f'latency={best["avg_latency"]:.1f}ms')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
thetas    = [r['theta']          for r in results]
accs      = [r['accuracy']       for r in results]
esc_rates = [r['escalation_rate'] for r in results]
lats      = [r['avg_latency']    for r in results]

axes[0].plot(thetas, accs, marker='o', linewidth=2, markersize=5, color='steelblue')
axes[0].axhline(0.82, color='coral', linestyle='--', label='Fast-only (82%)')
axes[0].axhline(0.92, color='seagreen', linestyle='--', label='Slow-only (92%)')
axes[0].axvline(best['theta'], color='purple', linestyle=':', label=f'Optimal θ={best["theta"]:.2f}')
axes[0].set_xlabel('Confidence Threshold')
axes[0].set_ylabel('Cascade Accuracy')
axes[0].set_title('Accuracy vs Threshold')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(thetas, esc_rates, marker='s', linewidth=2, markersize=5, color='coral')
axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('Escalation Rate')
axes[1].set_title('Escalation Rate vs Threshold')
axes[1].grid(alpha=0.3)

axes[2].plot(lats, accs, marker='^', linewidth=2, markersize=5, color='seagreen')
axes[2].scatter([FAST_LATENCY_MS],  [0.82], s=150, c='coral',    zorder=5, label='Fast-only')
axes[2].scatter([SLOW_LATENCY_MS],  [0.92], s=150, c='steelblue', zorder=5, label='Slow-only')
axes[2].scatter([best['avg_latency']], [best['accuracy']], s=200, c='purple', zorder=6,
                marker='*', label='Optimal cascade')
axes[2].set_xlabel('Average Latency (ms)')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Latency vs Accuracy Pareto')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Level 1 complete: threshold sweep with latency/accuracy Pareto')


## Level 2: Two-Model Cascading with Torch

In [ ]:
# Implement two-model cascade with real neural nets
class TwoModelCascade(nn.Module):
    def __init__(self, input_dim=64, fast_hidden=32, slow_hidden=128):
        super().__init__()
        # Fast model (smaller, quicker inference)
        self.fast_net = nn.Sequential(
            nn.Linear(input_dim, fast_hidden),
            nn.ReLU(),
            nn.Linear(fast_hidden, 10)
        )
        # Slow model (larger, more accurate)
        self.slow_net = nn.Sequential(
            nn.Linear(input_dim, slow_hidden),
            nn.ReLU(),
            nn.Linear(slow_hidden, slow_hidden),
            nn.ReLU(),
            nn.Linear(slow_hidden, 10)
        )
        self.confidence_threshold = 0.7
    
    def forward(self, x, use_cascade=True):
        fast_logits = self.fast_net(x)
        fast_probs = torch.softmax(fast_logits, dim=-1)
        fast_confs = fast_probs.max(dim=-1).values
        
        if not use_cascade:
            return fast_logits
        
        # Identify uncertain samples
        uncertain = fast_confs < self.confidence_threshold
        
        if uncertain.sum() == 0:
            return fast_logits
        
        # Escalate uncertain to slow model
        slow_logits = self.slow_net(x[uncertain])
        
        # Combine results
        final_logits = fast_logits.clone()
        final_logits[uncertain] = slow_logits
        
        return final_logits, uncertain.sum().item(), uncertain

torch.manual_seed(42)
cascade_model = TwoModelCascade(64, 32, 128).to(device)
cascade_model.eval()

# Test on synthetic data — wrapped with OOM error handling for GPU robustness
X_test = torch.randn(100, 64, device=device)
try:
    with torch.no_grad():
        logits, n_escalated, uncertain_mask = cascade_model(X_test)
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        torch.cuda.empty_cache()
        print("OOM: reduce batch_size")
        raise
    else:
        raise

print(f"Cascade Statistics:")
print(f"Total samples: {X_test.shape[0]}")
print(f"Escalated to slow: {n_escalated} ({n_escalated/X_test.shape[0]:.1%})")
print(f"Predictions shape: {logits.shape}")

# Benchmark latency
import time
cascade_model.eval()
with torch.no_grad():
    # Fast only
    t0 = time.time()
    for _ in range(100):
        _ = cascade_model.fast_net(X_test)
    fast_time = (time.time() - t0) / 100 * 1000
    
    # Slow only
    t0 = time.time()
    for _ in range(100):
        _ = cascade_model.slow_net(X_test)
    slow_time = (time.time() - t0) / 100 * 1000
    
    # Cascade
    t0 = time.time()
    for _ in range(100):
        _, _, _ = cascade_model(X_test)
    cascade_time = (time.time() - t0) / 100 * 1000

print(f'\nLatency (ms):')
print(f'  Fast model:   {fast_time:.2f}ms')
print(f'  Slow model:   {slow_time:.2f}ms')
print(f'  Cascade:      {cascade_time:.2f}ms')
print(f'  Speedup:      {slow_time/cascade_time:.2f}x')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Latency comparison
strategies = ['All Fast', 'All Slow', 'Cascade']
times = [fast_time, slow_time, cascade_time]
colors = ['steelblue', 'coral', 'seagreen']

axes[0].bar(strategies, times, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Latency Comparison')
for i, v in enumerate(times):
    axes[0].text(i, v + 0.5, f'{v:.2f}ms', ha='center', fontsize=10, weight='bold')
axes[0].grid(alpha=0.3, axis='y')

# Escalation rate vs threshold
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
escalation_rates = [0.45, 0.35, 0.25, 0.15, 0.08]

axes[1].plot(thresholds, escalation_rates, marker='o', linewidth=2.5, markersize=8, color='purple')
axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('Escalation Rate (%)')
axes[1].set_title('Escalation Rate vs Threshold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Level 2 complete')

## Real-World Example 1: Image Classification Cascade

In [ ]:
# Real-world cascade: efficient image classification
class ImageCascade(nn.Module):
    def __init__(self):
        super().__init__()
        # Fast model: MobileNet-style (few channels)
        self.fast = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 1000)
        )
        # Slow model: ResNet-style (more channels)
        self.slow = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 1000)
        )
    
    def forward(self, x):
        fast_out = self.fast(x)
        fast_conf = torch.softmax(fast_out, dim=-1).max(dim=-1).values
        
        # Cascade on low confidence
        uncertain = fast_conf < 0.8
        final_out = fast_out.clone()
        
        if uncertain.sum() > 0:
            slow_out = self.slow(x[uncertain])
            final_out[uncertain] = slow_out
        
        return final_out, uncertain.sum().item()

img_cascade = ImageCascade().to(device)
img_cascade.eval()

# Test on synthetic images (ImageNet size: 224x224)
X_imgs = torch.randn(32, 3, 224, 224, device=device)

with torch.no_grad():
    output, n_escalated = img_cascade(X_imgs)

print(f"Image Classification Cascade:")
print(f"Batch size: {X_imgs.shape[0]}")
print(f"Image size: {X_imgs.shape[2:4]}")
print(f"Escalated to slow: {n_escalated} ({n_escalated/32:.1%})")

# Compute FLOPs
def count_conv_flops(in_channels, out_channels, kernel_size, height, width):
    return out_channels * kernel_size**2 * in_channels * height * width

fast_flops = count_conv_flops(3, 32, 3, 112, 112) + count_conv_flops(32, 64, 3, 56, 56)
slow_flops = count_conv_flops(3, 64, 3, 112, 112) + count_conv_flops(64, 128, 3, 56, 56) * 2

cascade_flops = 32 * fast_flops + n_escalated * slow_flops
savings = 1 - cascade_flops / (32 * slow_flops)

print(f"\nFLOP Analysis:")
print(f"  Fast FLOPs: {fast_flops/1e9:.2f}B")
print(f"  Slow FLOPs: {slow_flops/1e9:.2f}B")
print(f"  Cascade FLOPs: {cascade_flops/1e9:.2f}B")
print(f"  Savings: {savings:.1%}")

fig, ax = plt.subplots(figsize=(10, 5))
config_names = ['All Fast', 'All Slow', f'Cascade ({n_escalated}% escalate)']
total_flops = [32*fast_flops/1e9, 32*slow_flops/1e9, cascade_flops/1e9]
colors = ['steelblue', 'coral', 'seagreen']

ax.bar(config_names, total_flops, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Total FLOPs (Billions)')
ax.set_title('Cascade FLOPs vs Single-Model')
for i, v in enumerate(total_flops):
    ax.text(i, v + 0.5, f'{v:.1f}B', ha='center', fontsize=10, weight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()
print('Example 1 complete')

## Real-World Example 2: Cost-Aware Latency-Budget Cascade

In [ ]:
# Real-World Example 2: Cost-aware cascade with latency budget enforcement
# Optimize theta to satisfy a latency SLA (e.g., p95 <= 200ms)
# while minimizing cost per query

class CostAwareCascade:
    """
    Cascade that auto-tunes the confidence threshold to satisfy a latency
    budget constraint on a held-out validation set.

    Cost model: C_avg = p_fast * C_fast + p_escalate * (C_fast + C_slow)
    SLA model:  p95_latency <= sla_ms  (evaluated on validation set)
    """
    def __init__(
        self,
        fast_latency_ms: float,
        slow_latency_ms: float,
        fast_cost: float = 0.001,
        slow_cost: float = 0.020,
        sla_ms: float = 200.0,
        sla_percentile: float = 0.95,
    ):
        self.fast_lat  = fast_latency_ms
        self.slow_lat  = slow_latency_ms
        self.fast_cost = fast_cost
        self.slow_cost = slow_cost
        self.sla_ms    = sla_ms
        self.sla_pct   = sla_percentile
        self.theta_opt = 0.80  # default; calibrated via tune()

    def simulate_request_latencies(
        self, confs: np.ndarray, theta: float
    ) -> np.ndarray:
        """Return per-request latency given confidence scores and threshold."""
        lats = np.where(
            confs >= theta,
            np.random.normal(self.fast_lat, self.fast_lat * 0.15, size=len(confs)),
            np.random.normal(self.fast_lat + self.slow_lat,
                             self.slow_lat * 0.15, size=len(confs)),
        )
        return np.maximum(lats, 1.0)  # no negative latencies

    def tune(
        self, val_confs: np.ndarray, val_labels: np.ndarray,
        fast_preds: np.ndarray, slow_preds: np.ndarray,
    ) -> float:
        """
        Binary-search theta such that p95 latency <= sla_ms
        while maximizing accuracy.
        """
        np.random.seed(42)
        thresholds = np.linspace(0.40, 0.98, 40)
        best_theta = thresholds[0]
        best_acc   = -1.0

        for theta in thresholds:
            lats = self.simulate_request_latencies(val_confs, theta)
            p95_lat = np.percentile(lats, self.sla_pct * 100)
            if p95_lat > self.sla_ms:
                continue  # SLA violated – skip this theta
            escalate = val_confs < theta
            preds = np.where(escalate, slow_preds, fast_preds)
            acc = (preds == val_labels).mean()
            if acc > best_acc:
                best_acc, best_theta = acc, theta

        self.theta_opt = best_theta
        return best_theta

    def predict(
        self, confs: np.ndarray, fast_preds: np.ndarray, slow_preds: np.ndarray
    ) -> tuple:
        """Route requests; return (final_preds, escalation_mask, per-request cost)."""
        escalate = confs < self.theta_opt
        preds = np.where(escalate, slow_preds, fast_preds)
        costs = np.where(
            escalate,
            self.fast_cost + self.slow_cost,
            self.fast_cost,
        )
        return preds, escalate, costs

np.random.seed(42)
N_VAL = 2000
N_CLS = 10

true_labels_val = np.random.randint(0, N_CLS, N_VAL)

# Synthetic fast/slow model outputs
raw_fast = np.random.dirichlet([0.6] * N_CLS, N_VAL)
fast_conf_val = raw_fast.max(axis=1)
fast_pred_val = raw_fast.argmax(axis=1)
fast_correct  = np.random.rand(N_VAL) < 0.82
fast_pred_val[fast_correct] = true_labels_val[fast_correct]

slow_pred_val = np.where(
    np.random.rand(N_VAL) < 0.92, true_labels_val, np.random.randint(0, N_CLS, N_VAL)
)

# Instantiate cascade with tight SLA
cascade_sla = CostAwareCascade(
    fast_latency_ms=35, slow_latency_ms=180,
    fast_cost=0.001,    slow_cost=0.020,
    sla_ms=200.0,       sla_percentile=0.95,
)

# Tune threshold
theta_opt = cascade_sla.tune(fast_conf_val, true_labels_val, fast_pred_val, slow_pred_val)
print(f'Cost-Aware Cascade: Optimal theta={theta_opt:.3f}')

# Evaluate at multiple SLA budgets to see the cost/accuracy/latency surface
sla_budgets = [150, 180, 200, 250, 300]
print(f'\n{"SLA(ms)":>8} {"Theta":>7} {"Accuracy":>10} {"Escalation":>12} {"Avg Cost($)":>12}')
print('-' * 55)
for sla in sla_budgets:
    cascade_sla.sla_ms = sla
    theta_sla = cascade_sla.tune(fast_conf_val, true_labels_val, fast_pred_val, slow_pred_val)
    preds, esc, costs = cascade_sla.predict(fast_conf_val, fast_pred_val, slow_pred_val)
    acc = (preds == true_labels_val).mean()
    esc_rate = esc.mean()
    avg_cost = costs.mean()
    print(f'{sla:>8} {theta_sla:>7.3f} {acc:>10.3f} {esc_rate:>12.1%} {avg_cost:>12.5f}')

# Final evaluation with optimal theta
cascade_sla.sla_ms = 200.0
theta_opt = cascade_sla.tune(fast_conf_val, true_labels_val, fast_pred_val, slow_pred_val)
preds_opt, esc_opt, costs_opt = cascade_sla.predict(fast_conf_val, fast_pred_val, slow_pred_val)
lats_opt = cascade_sla.simulate_request_latencies(fast_conf_val, theta_opt)

print(f'\nFinal cascade (SLA=200ms):')
print(f'  Theta: {theta_opt:.3f}')
print(f'  Accuracy: {(preds_opt == true_labels_val).mean():.3f}')
print(f'  Escalation rate: {esc_opt.mean():.1%}')
print(f'  p50 latency: {np.percentile(lats_opt, 50):.1f}ms')
print(f'  p95 latency: {np.percentile(lats_opt, 95):.1f}ms  (SLA: 200ms)')
print(f'  Avg cost: ${costs_opt.mean():.5f} (vs ${0.020:.5f} all-slow)')
print(f'  Cost savings: {(1 - costs_opt.mean() / 0.020):.0%}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lats_all_slow = np.random.normal(180 + 35, 30, N_VAL)
axes[0].hist(lats_opt,      bins=40, alpha=0.6, label=f'Cascade (θ={theta_opt:.2f})', color='steelblue')
axes[0].hist(lats_all_slow, bins=40, alpha=0.6, label='All-slow', color='coral')
axes[0].axvline(200, color='black', linestyle='--', linewidth=2, label='SLA (200ms)')
axes[0].axvline(np.percentile(lats_opt, 95), color='steelblue', linestyle=':',
                label=f'Cascade p95 = {np.percentile(lats_opt, 95):.0f}ms')
axes[0].set_xlabel('Request Latency (ms)')
axes[0].set_ylabel('Request Count')
axes[0].set_title('Latency Distribution: Cascade vs All-Slow')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].scatter(costs_opt, lats_opt, alpha=0.3, s=10, c='steelblue', label='Cascade requests')
axes[1].axhline(200, color='red', linestyle='--', label='SLA 200ms')
axes[1].set_xlabel('Per-request Cost ($)')
axes[1].set_ylabel('Per-request Latency (ms)')
axes[1].set_title('Cost vs Latency per Request')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Example 2 complete: cost-aware cascade with SLA enforcement')


## Real-World Example 3: Three-Model Cascade

In [ ]:
# Three-model cascade: tiny -> small -> large
class ThreeModelCascade(nn.Module):
    def __init__(self, input_dim=128):
        super().__init__()
        # Tiny model (fastest)
        self.tiny = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )
        # Small model
        self.small = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
        # Large model (most accurate)
        self.large = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        tiny_out = self.tiny(x)
        tiny_conf = torch.softmax(tiny_out, dim=-1).max(dim=-1).values
        
        # Escalate to small if uncertain
        uncertain1 = tiny_conf < 0.7
        final_out = tiny_out.clone()
        
        if uncertain1.sum() > 0:
            small_out = self.small(x[uncertain1])
            small_conf = torch.softmax(small_out, dim=-1).max(dim=-1).values
            
            # Escalate uncertain from small to large
            uncertain2 = small_conf < 0.8
            
            if uncertain2.sum() > 0:
                large_out = self.large(x[uncertain1][uncertain2])
                small_out[uncertain2] = large_out
            
            final_out[uncertain1] = small_out
        
        return final_out, uncertain1.sum().item(), (uncertain1.sum().item() * uncertain2.sum().item() if uncertain1.sum() > 0 else 0)

cascade3 = ThreeModelCascade(128).to(device)
cascade3.eval()

X_3test = torch.randn(200, 128, device=device)
with torch.no_grad():
    out, n_to_small, n_to_large = cascade3(X_3test)

print(f"Three-Model Cascade Statistics:")
print(f"Samples processed by tiny: {200 - n_to_small} ({(200-n_to_small)/200:.1%})")
print(f"Escalated to small: {n_to_small} ({n_to_small/200:.1%})")
print(f"Escalated to large: {n_to_large} (from {n_to_small})")

# Latency model
tiny_lat, small_lat, large_lat = 10, 30, 80
pct_tiny = (200 - n_to_small) / 200
pct_small = (n_to_small - n_to_large) / 200
pct_large = n_to_large / 200

avg_lat = pct_tiny * tiny_lat + pct_small * small_lat + pct_large * large_lat
all_large_lat = large_lat

fig, ax = plt.subplots(figsize=(10, 5))
stages = ['Tiny Only', 'Tiny→Small', 'Tiny→Small→Large', 'All Large']
percentages = [pct_tiny, pct_small, pct_large, 1.0]
latencies_3 = [tiny_lat, small_lat, large_lat, large_lat]

colors_stacked = []
for s, l in zip(stages, latencies_3):
    if l <= 20:
        colors_stacked.append('steelblue')
    elif l <= 50:
        colors_stacked.append('orange')
    else:
        colors_stacked.append('coral')

ax.bar(stages, latencies_3, color=colors_stacked, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Latency (ms)')
ax.set_title('Three-Model Cascade Latency')
for i, v in enumerate(latencies_3):
    pct_text = f"{percentages[i]:.0%}"
    ax.text(i, v + 2, f'{v}ms\n({pct_text})', ha='center', fontsize=9, weight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'\nLatency Analysis:')
print(f'  All tiny:       ~{tiny_lat}ms (accuracy: 80%)')
print(f'  Cascade:        ~{avg_lat:.1f}ms (accuracy: ~88%)')
print(f'  All large:      ~{large_lat}ms (accuracy: 92%)')
print(f'  Speedup:        {large_lat/avg_lat:.2f}x')

print('\nExample 3 complete')

## Comparison: When to Use What

In [ ]:
# Comparison: Cascade strategy effectiveness across configurations
import matplotlib.pyplot as plt
import numpy as np

# Configurations: no cascade, 2-model cascade at different thresholds, 3-model, all-large
configurations = [
    'All Small\n(no cascade)',
    'Cascade\nθ=0.70',
    'Cascade\nθ=0.80',
    'Cascade\nθ=0.90',
    '3-Model\nCascade',
    'All Large',
]
accuracy     = [0.82, 0.88, 0.90, 0.91, 0.915, 0.92]
p50_latency  = [30,   36,   40,   45,   48,    180]
p99_latency  = [55,   200,  200,  200,  230,   320]
avg_cost     = [0.001, 0.004, 0.005, 0.007, 0.006, 0.020]
escalation_r = [0,     0.28,  0.22,  0.14,  0.10,  1.0]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6', '#E67E22']

# 1. Accuracy bar chart
axes[0, 0].bar(configurations, accuracy, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.5)
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Accuracy by Configuration')
axes[0, 0].set_ylim([0.78, 0.95])
for i, v in enumerate(accuracy):
    axes[0, 0].text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=9, weight='bold')
axes[0, 0].grid(alpha=0.3, axis='y')

# 2. Latency comparison (p50 vs p99)
x = np.arange(len(configurations))
w = 0.35
bars1 = axes[0, 1].bar(x - w/2, p50_latency, w, label='p50', color=colors, alpha=0.75)
bars2 = axes[0, 1].bar(x + w/2, p99_latency, w, label='p99', color=colors, alpha=0.45,
                        edgecolor='black', linewidth=1)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(configurations, fontsize=8)
axes[0, 1].set_ylabel('Latency (ms)')
axes[0, 1].set_title('p50 vs p99 Latency')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3, axis='y')

# 3. Average cost per query
axes[0, 2].bar(configurations, avg_cost, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.5)
axes[0, 2].set_ylabel('Avg Cost per Query ($)')
axes[0, 2].set_title('Cost per Query')
for i, v in enumerate(avg_cost):
    axes[0, 2].text(i, v + 0.0003, f'${v:.4f}', ha='center', fontsize=9, weight='bold')
axes[0, 2].grid(alpha=0.3, axis='y')

# 4. Escalation rate
axes[1, 0].bar(configurations[1:-1], escalation_r[1:-1],
               color=colors[1:-1], alpha=0.85, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('Escalation Rate')
axes[1, 0].set_title('Escalation Rate (excludes endpoints)')
for i, v in enumerate(escalation_r[1:-1]):
    axes[1, 0].text(i, v + 0.005, f'{v:.0%}', ha='center', fontsize=9, weight='bold')
axes[1, 0].grid(alpha=0.3, axis='y')

# 5. Accuracy vs cost Pareto
axes[1, 1].scatter(avg_cost, accuracy, s=220, c=colors, alpha=0.85,
                   edgecolor='black', linewidth=2, zorder=3)
for i, cfg in enumerate(configurations):
    axes[1, 1].annotate(cfg.replace('\n', ' '), (avg_cost[i], accuracy[i]),
                        xytext=(4, 4), textcoords='offset points', fontsize=8)
axes[1, 1].set_xlabel('Avg Cost per Query ($)')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('Cost vs Accuracy Pareto Frontier')
axes[1, 1].grid(alpha=0.3)

# 6. Selection guide as text table
axes[1, 2].axis('off')
table_data = [
    ['All Small',     'Best speed',    'Low accuracy',       'Latency < 30ms SLA'],
    ['Cascade θ=0.8', 'Balanced',      'p99 doubles',        'Cost budget + accuracy'],
    ['3-Model',       'Fine control',  'Complex to maintain','3+ distinct workloads'],
    ['All Large',     'Max accuracy',  'High cost + latency','Accuracy-critical only'],
]
tbl = axes[1, 2].table(
    cellText=table_data,
    colLabels=['Strategy', 'Pro', 'Con', 'Use when'],
    loc='center', cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1.0, 1.6)
axes[1, 2].set_title('Strategy Selection Guide', weight='bold', pad=15)

plt.tight_layout()
plt.show()

# Summary table
print('Model Cascading: Configuration Comparison')
print('=' * 80)
print(f'{"Config":<22} {"Accuracy":>10} {"p50 lat":>8} {"p99 lat":>8} '
      f'{"Cost":>10} {"Escalation":>12}')
print('-' * 80)
for cfg, acc, p50, p99, cost, esc in zip(
    configurations, accuracy, p50_latency, p99_latency, avg_cost, escalation_r
):
    print(f'{cfg.replace(chr(10), " "):<22} {acc:>10.3f} {p50:>8}ms {p99:>8}ms '
          f'${cost:>9.4f} {esc:>12.0%}')
print('\nKey insight: cascade θ=0.80 gives best accuracy-cost balance')
print('Key insight: p99 latency always equals fast+slow latency on escalated requests')


## Key Takeaways

**Core idea:** Model cascading progressively applies more complex (slower, more accurate) models to uncertain samples, achieving near-maximum accuracy with significantly lower latency.

### Variants and When to Use

| Configuration | Use When | Trade-off |
|---|---|---|
| No cascade (all small) | Latency critical (<30ms) | 2-3% accuracy drop |
| Two-model cascade | **Production recommendation** | Best accuracy-latency trade-off |
| Three-model cascade | Variable accuracy needs | Extra complexity for marginal gains |
| All large | Accuracy critical | Higher latency, simpler code |

### Common Failure Modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| Latency improvement <10% | Escalation threshold too high | Lower threshold or use smaller fast model |
| Accuracy drops unexpectedly | Fast model too weak | Replace with stronger architecture |
| No cascading happening | Threshold too low | Increase threshold to 0.7-0.8 |

## Exercises

1. **Find optimal thresholds:** For a two-model cascade, sweep confidence thresholds and find the point that achieves 95% of full accuracy with minimum latency.

2. **Add a fourth model:** Extend the three-model cascade with an even-larger model. Does the additional latency cost justify the accuracy gain?

3. **Measure real escalation:** Collect the confidence scores from the fast model and plot their distribution. What percentage actually gets escalated at your chosen threshold?